In [1]:
# Pretty inline figures + autoreload
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'


In [2]:
# If you run the notebook on farm nodes, you can keep prints compact
import os
os.environ["PYTHONWARNINGS"] = "ignore"

# Point Python to your VQniche src so the local imports work
import sys
sys.path.insert(0, "/nfs/team361/mv11/vqniche/src")  # <<< make sure this matches your VQniche src root
# /nfs/team361/mv11/vqniche/src/vqniche 
# Standard libs
from pathlib import Path
from pprint import pprint

# Core scientific stack
import numpy as np
import torch

# AnnData/PyG + your project modules
from vqniche.dataset.in_memory_dataset_blob import InMemoryDatasetBlob
from vqniche.dataset.transforms import (
    init_gene_count_transforms,
    SetExperimentDataKeys,
)

# Quick guardrails to surface device info
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("LSF cores:", os.environ.get("LSB_DJOB_NUMPROC", "1"))


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: libcudart.so.11.0: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_cluster/_version_cuda.so: unde

CUDA available: True
CUDA device count: 1
LSF cores: 6


### dataset roots

In [3]:
from pathlib import Path

# Top-level datasets directory on NFS
DATA_DIR = Path("/nfs/team361/mv11/DATASETS")

# <<< only this line differs >>>
DATASET_NAME = "merfish_mouse_cortex_from_luna_csv__degraded"

SILVER_DIR = DATA_DIR / "silver" / DATASET_NAME
GOLD_DIR   = DATA_DIR / "gold" / "in-memory-PyG-dataset-blob" / DATASET_NAME

print("Expect silver batches under:", SILVER_DIR)
print("Gold will be written to   :", GOLD_DIR)

silver_files = list(SILVER_DIR.glob("**/*.h5ad"))
print(f"Found {len(silver_files)} silver batch file(s).")
for f in silver_files[:10]:
    print("  •", f.name)

assert len(silver_files) > 0, f"No .h5ad files in {SILVER_DIR}"


Expect silver batches under: /nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv__degraded
Gold will be written to   : /nfs/team361/mv11/DATASETS/gold/in-memory-PyG-dataset-blob/merfish_mouse_cortex_from_luna_csv__degraded
Found 64 silver batch file(s).
  • merfish_mouse_cortex_mouse2_slice119.h5ad
  • merfish_mouse_cortex_mouse2_slice209.h5ad
  • merfish_mouse_cortex_mouse2_slice129.h5ad
  • merfish_mouse_cortex_mouse1_slice313.h5ad
  • merfish_mouse_cortex_mouse2_slice270.h5ad
  • merfish_mouse_cortex_mouse2_slice40.h5ad
  • merfish_mouse_cortex_mouse2_slice151.h5ad
  • merfish_mouse_cortex_mouse2_slice189.h5ad
  • merfish_mouse_cortex_mouse1_slice102.h5ad
  • merfish_mouse_cortex_mouse2_slice79.h5ad


In [4]:
from pathlib import Path
import torch

from vqniche.dataset.in_memory_dataset_blob import InMemoryDatasetBlob
from vqniche.dataset.transforms import SetExperimentDataKeys
from vqniche.dataloaders.in_memory_datamodule import InMemoryDataModule
from torch_geometric.data import Batch

feature_names = ["cell_gene_counts"]
label_names   = ["cell_types=cell_type"]

graph_kwargs = dict(
    coord_type="generic",
    spatial_key="spatial",
    include_self_loop=False,
    n_neighs_list=[10],
    radius_list=None,
    k={},
)


/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [5]:
software_paths = dict(deepwalk="", gosh="")

# First build of this namespace -> True. After it succeeds, set to False.
OVERWRITE = True

dataset_blob = InMemoryDatasetBlob(
    name=DATASET_NAME,
    feature_names=feature_names,
    label_names=label_names,
    graph_kwargs=graph_kwargs,
    data_directory_path=DATA_DIR,
    pre_transform=None,
    pre_filter=None,
    overwrite=OVERWRITE,
    software_paths=software_paths,
)

print("processed_dir:", dataset_blob.processed_dir)


Processing...


All batches have the same gene panel.
Label Name: cell_types | self.label_categories[label_name]=['Astro', 'Endo', 'L2/3 IT', 'L4/5 IT', 'L5 ET', 'L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6 IT Car3', 'L6b', 'Lamp5', 'Micro', 'OPC', 'Oligo', 'PVM', 'Peri', 'Pvalb', 'SMC', 'Sncg', 'Sst', 'VLMC', 'Vip']
Number of Cores: 6 | Number of Tissue Sections: 64
Processing /nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv__degraded/merfish_mouse_cortex_mouse2_slice119.h5ad...
Computing spatial neighbors with delaunay=False, radius=None, n_neighs=10 at key spatial_n_neighs_10...Computing spatial neighbors with delaunay=False, radius=None, n_neighs=10 at key spatial_n_neighs_10...

Computing spatial neighbors with delaunay=False, radius=None, n_neighs=10 at key spatial_n_neighs_10...Computing spatial neighbors with delaunay=False, radius=None, n_neighs=10 at key spatial_n_neighs_10...
Computing spatial neighbors with delaunay=False, radius=None, n_neighs=10 at key spatial_n_neighs_1

Done!
/nfs/team361/mv11/.venvs/LUNA/lib/python3.10/site-packages/torch_geometric/io/fs.py:229: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "


processed_dir: /nfs/team361/mv11/DATASETS/gold/in-memory-PyG-dataset-blob/merfish_mouse_cortex_from_luna_csv__degraded


# 📝 What we did (quick recap to keep next to the cell)

* **Notebook location:** `scgg-reproducibility/experiments/VQNiche/00_silver_to_gold_merfish.ipynb`
* **Kernel:** using your `LUNA (py3.10-cu118)` env is fine for preprocessing (PyG CUDA ext warnings are harmless).
* **Codebase path:** added `/nfs/team361/mv11/vqniche/src` to `sys.path`.
* **Where the data lives:**

  * **Silver (input .h5ad):** `/nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv`
  * **Gold (output blob):** `/nfs/team361/mv11/DATASETS/gold/in-memory-PyG-dataset-blob/merfish_mouse_cortex_from_luna_csv`
  * Staying on **NFS** (stable); no Lustre needed for this step.
* **Features/labels picked:**

  * `cell_gene_counts` → raw cell×gene counts (254 genes).
  * `cell_types` ← `adata.obs['cell_type']` (23 categories).
* **Graph:** built **kNN (k=10)** over `obsm['spatial']` with Squidpy coordinate type **"generic"** (not “cartesian”).
  It writes an undirected edge list per batch (e.g. `spatial_n_neighs_10.edgelist`).
* **Processing result:** 64 tissue sections processed. Blob + metadata saved under the gold dir.
  You’ll re-load by re-instantiating `InMemoryDatasetBlob` with `overwrite=False`.

In [7]:
probe0 = dataset_blob[0]
edge_keys = [k for k in probe0.keys() if k.startswith("edge_index_")]
edge_index_name = edge_keys[0].replace("edge_index_", "")
print("edge_index_name ->", edge_index_name)


edge_index_name -> spatial_n_neighs_10


In [8]:
from vqniche.dataset.transforms import SetExperimentDataKeys

set_keys = SetExperimentDataKeys(
    feature_names=["X"],
    label_name="cell_types",
    edge_index_name=edge_index_name,
)

dataset_for_exp = InMemoryDatasetBlob(
    name=DATASET_NAME,
    feature_names=feature_names,
    label_names=label_names,
    graph_kwargs=graph_kwargs,
    data_directory_path=DATA_DIR,
    overwrite=False,           # now just load
    software_paths=software_paths,
    transform=set_keys,
)

len(dataset_for_exp), dataset_for_exp.processed_dir


print("Dataset ready. Batches:", len(dataset_for_exp))


Dataset ready. Batches: 64


In [9]:
import pickle
from pathlib import Path

with open(Path(dataset_for_exp.processed_dir) / "label_categories.pkl", "rb") as f:
    label_categories = pickle.load(f)
print("cell_types classes:", label_categories["cell_types"])

with open(Path(dataset_for_exp.processed_dir) / "gene_panel.pkl", "rb") as f:
    gene_panel = pickle.load(f)
print("genes:", gene_panel.shape[0], "| head:\n", gene_panel.head())


cell_types classes: ['Astro', 'Endo', 'L2/3 IT', 'L4/5 IT', 'L5 ET', 'L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6 IT Car3', 'L6b', 'Lamp5', 'Micro', 'OPC', 'Oligo', 'PVM', 'Peri', 'Pvalb', 'SMC', 'Sncg', 'Sst', 'VLMC', 'Vip']
genes: 249 | head:
                        ensembl_id
gene_name                        
1700022I11Rik  ENSMUSG00000028451
5730522E02Rik  ENSMUSG00000032985
Acta2          ENSMUSG00000035783
Adam2          ENSMUSG00000022039
Adamts2        ENSMUSG00000036545


In [10]:
g = torch.Generator().manual_seed(42)
dataset_fixed = []

for d in dataset_for_exp:
    # these came from silver via your blob builder
    assert hasattr(d, "is_train_orig") and hasattr(d, "is_test_orig"), "Missing original split flags on Data."

    train_mask = d.is_train_orig.clone()
    val_mask   = torch.zeros_like(train_mask)  # no val
    test_mask  = d.is_test_orig.clone()

    # # OPTIONAL: carve ~10% of *train* as validation (seeded)
    # idx = torch.where(train_mask)[0]
    # if idx.numel() > 0:
    #     perm = idx[torch.randperm(idx.numel(), generator=g)]
    #     n_val = max(1, int(0.1 * len(perm)))
    #     val_mask[perm[:n_val]] = True
    #     train_mask[perm[:n_val]] = False

    d.train_mask, d.val_mask, d.test_mask = train_mask, val_mask, test_mask
    dataset_fixed.append(d)

print("Built LUNA-parity masks for", len(dataset_fixed), "sections.")


Built LUNA-parity masks for 64 sections.


In [11]:
tot_tr = sum(int(d.train_mask.sum()) for d in dataset_fixed)
tot_va = sum(int(d.val_mask.sum())   for d in dataset_fixed)
tot_te = sum(int(d.test_mask.sum())  for d in dataset_fixed)
print(f"TOTAL  train={tot_tr:,}  val={tot_va:,}  test={tot_te:,}")

assert all(((d.train_mask & d.test_mask) == 0).all() for d in dataset_fixed)
assert all(((d.val_mask   & d.test_mask) == 0).all() for d in dataset_fixed)
assert all(((d.train_mask & d.val_mask)  == 0).all() for d in dataset_fixed)
print("Mask overlaps: none ✅")


TOTAL  train=157,580  val=0  test=117,265
Mask overlaps: none ✅


In [12]:
data_all = Batch.from_data_list(dataset_fixed)
# Provide section ids so the DataModule can see them
data_all.adata_batch_ids = data_all.batch  # int tensor
print("All nodes:", data_all.x.shape[0], "| features:", data_all.x.shape[1])


All nodes: 274845 | features: 249
